# Use cases

Three concrete lookups on top of the mock data:

1. **per compound** — all targets it has been tested against
2. **per target** — all compounds tested against it
3. **per protein family** — which compounds are available for it

This connects to `probe.db`, the on-disk database built from `staging/_template`
by `examples/build_mock_db.py`. Run that script first if the file does not exist
yet:

```bash
uv run python examples/build_mock_db.py
```

In [1]:
from pathlib import Path

import pandas as pd

from probedb import ProbeDB

pd.set_option("display.max_colwidth", 44)
pd.set_option("display.width", 170)

DB_PATH = Path("..") / "probe.db"
assert DB_PATH.exists(), f"{DB_PATH} not found -- run examples/build_mock_db.py first"

db = ProbeDB(DB_PATH, create=False)

db.counts()

,table,rows
0,compound,3
1,chembl,3
2,uniprot,8
3,target,6
4,target_uniprot,9
5,bioactivity_source,10
6,bioactivity_group,8
7,bioactivity,13


## Use case 1: per compound

For every compound, answer four questions:

- **which targets** has it been measured against?
- **which set(s)** do the measurements come from (`source_db`: opnMe,
  Probes & Drugs, in-house, literature, ...)?
- **what is its main target** — the one with the strongest reported potency?
- **what is its selectivity** — how much weaker is the next best target on the
  same scale?

"Main target" and "selectivity" only compare rows that are actually
comparable: `bioactivity_type == "IC50"`, `unit == "nM"`, `relation == "="`.
Mixing pIC50 on -log(M) or a cell EC50 into the same ranking as a biochemical
IC50 would be comparing different things, so those stay out of the ranking.
Anything measured as a bound (`>`, `<`, ...) is a counter-screen, not a
potency, so it is reported separately and read qualitatively.

In [2]:
def compound_profile(db, compound):
    hits = db.bioactivities(compound=compound)

    targets = hits[["target_type", "target"]].drop_duplicates().reset_index(drop=True)
    sources = sorted(hits["source_db"].dropna().unique())

    comparable = hits[
        (hits.bioactivity_type == "IC50") & (hits.unit == "nM") & (hits.relation == "=")
    ]
    potency = (
        comparable.groupby(["target", "target_type"], as_index=False)["value"]
        .median()
        .sort_values("value")
        .reset_index(drop=True)
    )

    counter_screens = hits[hits.relation.isin([">", ">=", "<", "<="])]

    return targets, sources, potency, counter_screens


for compound in db.table("compound")["name"]:
    targets, sources, potency, counter_screens = compound_profile(db, compound)

    print(f"== {compound} ==")

    print(f"targets ({len(targets)}):")
    for _, row in targets.iterrows():
        print(f"  [{row.target_type}] {row.target}")

    print(f"derived from: {', '.join(sources) if sources else 'no source recorded'}")

    if potency.empty:
        print("main target: no comparable IC50 (nM, '=') data")
    else:
        best = potency.iloc[0]
        print(f"main target: {best.target}  (IC50 = {best.value:g} nM)")
        if len(potency) > 1:
            second = potency.iloc[1]
            fold = second.value / best.value
            print(
                f"selectivity: {fold:.1f}-fold vs {second.target} "
                f"(IC50 = {second.value:g} nM), the next best on the same scale"
            )
        else:
            print("selectivity: only one target with comparable IC50 (nM) data")

    if not counter_screens.empty:
        print("counter-screens (different scale, read qualitatively):")
        for _, row in counter_screens.iterrows():
            print(f"  {row.target}: {row.bioactivity_type} {row.relation} {row.value:g} {row.unit}")

    print()

== BI-2536 ==
targets (3):
  [protein] Serine/threonine-protein kinase PLK1
  [protein] Bromodomain-containing protein 4
  [complex] Cyclin-dependent kinase 1/cyclin B1
derived from: Probes & Drugs, in-house, literature, opnMe
main target: Serine/threonine-protein kinase PLK1  (IC50 = 0.965 nM)
selectivity: 1.2-fold vs Bromodomain-containing protein 4 (IC50 = 1.2 nM), the next best on the same scale
counter-screens (different scale, read qualitatively):
  Cyclin-dependent kinase 1/cyclin B1: IC50 > 10000 nM

== (+)-JQ1 ==
targets (2):
  [protein] Bromodomain-containing protein 4
  [protein] Bromodomain-containing protein 2
derived from: literature
main target: Bromodomain-containing protein 2  (IC50 = 130.75 nM)
selectivity: only one target with comparable IC50 (nM) data

== Olaparib ==
targets (3):
  [protein] Poly [ADP-ribose] polymerase 1
  [family] PARP 1, 2 and 3
  [protein] Serine/threonine-protein kinase PLK1
derived from: in-house, literature
main target: Poly [ADP-ribose] poly

## Use case 2: per target

The mirror image of use case 1. For every protein, complex and family, answer:

- **how many compounds** have been tested against it, and **which set(s)**
  do they come from?
- **which compound is most potent** — lowest comparable IC50?
- **which compound is most selective** — the one for which this target is the
  strongest hit by the widest margin, compared to that same compound's own
  next best target?

"Most selective" reuses `compound_profile` from use case 1: for each compound
tested here, it looks at *that compound's* full potency ranking across all its
targets and asks how this target compares to the compound's best *other*
target. A ratio above 1 means the compound genuinely prefers this target; a
ratio below 1 means even the best candidate here is actually more potent
somewhere else, so nothing tested is truly selective for it.

In [3]:
def target_profile(db, target_id):
    hits = db.bioactivities(target=target_id)

    compounds = sorted(hits["compound"].unique())
    sources = sorted(hits["source_db"].dropna().unique())

    comparable = hits[
        (hits.bioactivity_type == "IC50") & (hits.unit == "nM") & (hits.relation == "=")
    ]
    potency = (
        comparable.groupby("compound", as_index=False)["value"]
        .median()
        .sort_values("value")
        .reset_index(drop=True)
    )

    return compounds, sources, potency


def target_preference(db, compound, target_name):
    # how this target compares to `compound`'s own best *other* target,
    # reusing the per-compound potency ranking from use case 1
    _, _, potency, _ = compound_profile(db, compound)
    at_target = potency[potency.target == target_name]
    others = potency[potency.target != target_name]
    if at_target.empty or others.empty:
        return None
    return others.value.min() / at_target.value.iloc[0]


for _, target in db.table("target").iterrows():
    compounds, sources, potency = target_profile(db, target.target_id)
    name = target["name"]

    print(f"== {name} ({target.type}) ==")
    print(f"compounds tested ({len(compounds)}): {', '.join(compounds) if compounds else 'none'}")
    print(f"derived from: {', '.join(sources) if sources else 'no source recorded'}")

    if potency.empty:
        print("most potent: no comparable IC50 (nM, '=') data")
        print("most selective: not applicable, no comparable potency data")
    else:
        best = potency.iloc[0]
        print(f"most potent: {best.compound}  (IC50 = {best.value:g} nM)")

        ratios = [(c, target_preference(db, c, name)) for c in potency["compound"]]
        ratios = [(c, r) for c, r in ratios if r is not None]
        if not ratios:
            print("most selective: no compound has another comparable target to compare against")
        else:
            top_compound, top_ratio = max(ratios, key=lambda cr: cr[1])
            if top_ratio > 1:
                print(f"most selective: {top_compound}  ({top_ratio:.1f}-fold vs its next best target)")
            else:
                print(
                    f"most selective: none, really -- even {top_compound} is "
                    f"{1 / top_ratio:.1f}-fold more potent on a different target"
                )

    print()

== Serine/threonine-protein kinase PLK1 (protein) ==
compounds tested (2): BI-2536, Olaparib
derived from: Probes & Drugs, in-house, opnMe
most potent: BI-2536  (IC50 = 0.965 nM)
most selective: BI-2536  (1.2-fold vs its next best target)

== Bromodomain-containing protein 4 (protein) ==
compounds tested (2): (+)-JQ1, BI-2536
derived from: literature
most potent: BI-2536  (IC50 = 1.2 nM)
most selective: none, really -- even BI-2536 is 1.2-fold more potent on a different target

== Bromodomain-containing protein 2 (protein) ==
compounds tested (1): (+)-JQ1
derived from: literature
most potent: (+)-JQ1  (IC50 = 130.75 nM)
most selective: no compound has another comparable target to compare against

== Poly [ADP-ribose] polymerase 1 (protein) ==
compounds tested (1): Olaparib
derived from: literature
most potent: Olaparib  (IC50 = 0.1 nM)
most selective: Olaparib  (208.9-fold vs its next best target)

== PARP 1, 2 and 3 (family) ==
compounds tested (1): Olaparib
derived from: literature
m

## Use case 3: per protein family

For a family target -- RAS, PARP 1/2/3, whatever a source has grouped -- which
chemical probes, chemogenomic compounds or drugs are available?

A family is just `target.type == "family"`; `target_uniprot` says which
accessions belong to it (see the [Complexes](#Complexes) section of
`explore_db.ipynb` for the accession side of the same idea). This mock set has
one family target, `PARP 1, 2 and 3` -- no RAS family is loaded here, but the
same code runs unchanged on one, or on any other family, once its data is.

One honesty note: the schema has no `probe` / `chemogenomic` / `drug` column
-- `compound` only carries an InChIKey, a SMILES and a name (`database/schema.sql`).
So "which compounds are available" is answered with what actually is on file:
name, InChIKey, ChEMBL id (a real external identifier, when known) and the
`source_db` each measurement came from. `source_db` is the closest thing to a
provenance signal here -- `opnMe` denotes an open chemical probe, `Probes &
Drugs` a database that itself distinguishes probes from drugs -- but it is a
hint, not a classification field, and this notebook does not pretend
otherwise.

In [4]:
families = db.table("target")
families = families[families.type == "family"]
families

,target_id,type,name
4,5,family,"PARP 1, 2 and 3"


In [5]:
FAMILY_NAME = families.iloc[0]["name"]  # "PARP 1, 2 and 3" here -- swap in "RAS" or
                                         # any other family name once one is loaded
family_id = int(families.iloc[0].target_id)

compounds, sources, potency = target_profile(db, family_id)

identity = db.table("compound").merge(db.table("chembl"), on="inchikey", how="left")
identity = identity[identity["name"].isin(compounds)]

print(f"== {FAMILY_NAME} (family) ==")
print(f"compounds available ({len(compounds)}), derived from: {', '.join(sources)}")
print()

for _, row in identity.iterrows():
    chembl = row.chembl_id if pd.notna(row.chembl_id) else "no ChEMBL id on file"
    hit = potency[potency.compound == row["name"]]
    ic50 = f"{hit.value.iloc[0]:g} nM" if not hit.empty else "no comparable IC50 (nM, '=') data"
    print(row["name"])
    print(f"  inchikey: {row.inchikey}")
    print(f"  chembl:   {chembl}")
    print(f"  smiles:   {row.smiles[:40]}...")
    print(f"  IC50 on {FAMILY_NAME}: {ic50}")
    print()

== PARP 1, 2 and 3 (family) ==
compounds available (1), derived from: literature

Olaparib
  inchikey: FDLYAMZZIXQODN-UHFFFAOYSA-N
  chembl:   CHEMBL521686
  smiles:   O=C(c1cc(Cc2n[nH]c(=O)c3ccccc23)ccc1F)N1...
  IC50 on PARP 1, 2 and 3: 20.89 nM

